# Local Inference with Ollama

Needs `ollama serve` running.

- `list_models()` - what's installed
- `query_model(prompt, model)` - prompt in, reply out
- `unload(model)` - free it from memory

In [5]:
import requests

BASE = "http://localhost:11434"

def list_models():
    """Return the names of all locally installed models."""
    models = requests.get(f"{BASE}/api/tags").json()["models"]
    return [m["name"] for m in models]

list_models()

['qwen3.8:27b-mtp-q4_K_M', 'qwen3.8:27b-mlx']

In [6]:
def query_model(prompt, model, think=None):
    """Send a prompt to any local model and return its reply.

    think=False hides reasoning on reasoning models (e.g. qwen3);
    leave it as None for models that don't support thinking.
    """
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
    }
    if think is not None:
        payload["think"] = think
    r = requests.post(f"{BASE}/api/chat", json=payload, timeout=120)
    r.raise_for_status()
    return r.json()["message"]["content"]

In [7]:
def unload(model):
    """Free a model from memory (keep_alive=0 unloads it immediately)."""
    requests.post(f"{BASE}/api/generate", json={"model": model, "keep_alive": 0})

In [8]:
# Which models are loaded in memory vs just installed?
loaded = {m["name"] for m in requests.get(f"{BASE}/api/ps").json()["models"]}
for m in list_models():
    print("loaded    " if m in loaded else "unloaded  ", m)

unloaded   qwen3.8:27b-mtp-q4_K_M
loaded     qwen3.8:27b-mlx


In [9]:
import time

model = "qwen3.8:27b-mlx"  # pick any name from list_models()
start = time.perf_counter()
reply = query_model("What do llamas eat?", model, think=False)
print(reply)
print(f"\n[{model}: {time.perf_counter() - start:.1f}s]")

Llamas are herbivores, meaning their diet consists entirely of plant material. As ruminants (similar to cows and sheep), they have a complex stomach system that allows them to efficiently digest tough, fibrous plants. Here’s a breakdown of what llamas eat:

### 1. Primary Diet in the Wild/On Pasture
- **Forage:** Grasses, clover, alfalfa, and other leafy greens form the bulk of their diet.
- **Browsing:** Llamas are also browsers, meaning they eat from shrubs, bushes, and tree leaves. They are highly efficient eaters and can digest low-quality forage that other livestock might leave behind.
- **Adaptation:** Their diet is well-suited to the high-altitude, arid environments of the Andes, where they originally evolved. They can thrive on sparse vegetation.

### 2. Supplementary Feeds (in Captivity or Farm Settings)
- **Hay:** When fresh pasture is unavailable (e.g., in winter), llamas are typically fed high-quality hay, such as:
  - Grass hay
  - Alfalfa hay (in moderation, as it is high